## Universidad Autónoma de Aguascalientes
## Departamento: Ciencias de la Computación
## Carrera: Ingeniería en Computación Inteligente
## Curso: Machine y Deep Learning
## Maestro: Dr. Francisco Javier Luna Rosas
## Alumno: Guillermo González Lara (237864)
## Semestre: Enero-Junio del 2026

---

# PRÁCTICA: Análisis de Sentimientos con Redes Neuronales Convolucionales 1D (CNN-1D)

## Introducción

El **Análisis de Sentimientos** (o *Sentiment Analysis*) es una tarea de Procesamiento de Lenguaje Natural (NLP) que consiste en identificar y extraer información subjetiva de texto. En este caso, determinaremos si una crítica de cine tiene un sentimiento **positivo (1)** o **negativo (0)**.

### ¿Por qué CNN-1D para texto?

Aunque las CNNs son más conocidas por su aplicación en imágenes (2D), las **CNNs 1D** son una arquitectura muy eficiente para datos secuenciales como texto. Funcionan aplicando filtros convolucionales a lo largo de la secuencia de palabras (en lugar de sobre píxeles), capturando **patrones locales de n-gramas** (por ejemplo, pares o tríos de palabras que aparecen juntos) que son informativos para la clasificación.

Ventajas sobre RNNs/LSTMs:
- Entrenamiento más rápido (operaciones paralelizables)
- Captura eficientemente relaciones locales entre palabras
- Excelente rendimiento en tareas de clasificación de texto

### Dataset: IMDB Movie Reviews

Utilizaremos el dataset de reseñas de películas de IMDB, que contiene **50,000 reseñas** balanceadas entre sentimientos positivos y negativos. Keras incluye este dataset preprocesado con palabras ya codificadas como enteros.

## Paso 1: Importar las librerías necesarias

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, Conv1D, GlobalMaxPooling1D, MaxPooling1D,
    Dense, Dropout, Flatten, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

print("TensorFlow versión:", tf.__version__)
print("Keras versión:", keras.__version__)
print("NumPy versión:", np.__version__)

## Paso 2: Cargar y explorar el dataset (IMDB)

In [ ]:
# Hiperparámetro: tamaño del vocabulario (top N palabras más frecuentes)
VOCAB_SIZE = 10000

# Cargamos el dataset con las top 10,000 palabras más frecuentes
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

print("=== Dimensiones del dataset ===")
print(f"Muestras de entrenamiento : {len(X_train)}")
print(f"Muestras de prueba        : {len(X_test)}")
print(f"\nDistribución de clases en entrenamiento:")
print(f"  Positivos (1): {np.sum(y_train == 1)}")
print(f"  Negativos (0): {np.sum(y_train == 0)}")

print(f"\nEjemplo de reseña (como secuencia de enteros):")
print(X_train[0][:30], "... (longitud:", len(X_train[0]), "palabras)")
print(f"Etiqueta correspondiente  : {y_train[0]} ({'Positivo' if y_train[0] == 1 else 'Negativo'})")

## Paso 3: Análisis exploratorio de los datos

In [ ]:
# Analizamos la distribución de longitudes de las reseñas
train_lengths = [len(x) for x in X_train]
test_lengths  = [len(x) for x in X_test]

print("=== Estadísticas de longitud de reseñas (palabras) ===")
print(f"  Mínimo  : {np.min(train_lengths)}")
print(f"  Máximo  : {np.max(train_lengths)}")
print(f"  Media   : {np.mean(train_lengths):.1f}")
print(f"  Mediana : {np.median(train_lengths):.1f}")
print(f"  Percentil 90: {np.percentile(train_lengths, 90):.0f}")
print(f"  Percentil 95: {np.percentile(train_lengths, 95):.0f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histograma de longitudes
axes[0].hist(train_lengths, bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(np.mean(train_lengths), color='red', linestyle='--', label=f'Media ({np.mean(train_lengths):.0f})')
axes[0].axvline(np.percentile(train_lengths, 95), color='orange', linestyle='--',
                label=f'P95 ({np.percentile(train_lengths, 95):.0f})')
axes[0].set_title('Distribución de longitudes de reseñas', fontsize=13)
axes[0].set_xlabel('Número de palabras')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

# Distribución de clases
labels = ['Negativo (0)', 'Positivo (1)']
counts = [np.sum(y_train == 0), np.sum(y_train == 1)]
axes[1].bar(labels, counts, color=['#e74c3c', '#2ecc71'], edgecolor='white', width=0.5)
axes[1].set_title('Balance de clases (entrenamiento)', fontsize=13)
axes[1].set_ylabel('Número de reseñas')
for i, v in enumerate(counts):
    axes[1].text(i, v + 100, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## Paso 4: Decodificar y visualizar reseñas reales

El dataset IMDB almacena las palabras como índices enteros. Vamos a recuperar el diccionario inverso para ver las reseñas como texto legible.

In [ ]:
# Cargamos el índice de palabras del dataset
word_index = imdb.get_word_index()

# Creamos el diccionario inverso (índice → palabra)
# Nota: los primeros 3 índices están reservados por Keras
reverse_word_index = {v + 3: k for k, v in word_index.items()}
reverse_word_index[0] = '<PAD>'
reverse_word_index[1] = '<START>'
reverse_word_index[2] = '<UNK>'
reverse_word_index[3] = '<UNUSED>'

def decode_review(encoded_review):
    """Convierte una secuencia de enteros a texto legible."""
    return ' '.join([reverse_word_index.get(i, '?') for i in encoded_review])

# Mostramos ejemplos de reseñas positivas y negativas
print("=" * 70)
print("EJEMPLO DE RESEÑA POSITIVA (y=1):")
print("=" * 70)
pos_idx = np.where(y_train == 1)[0][0]
print(decode_review(X_train[pos_idx])[:500], "...")

print("\n" + "=" * 70)
print("EJEMPLO DE RESEÑA NEGATIVA (y=0):")
print("=" * 70)
neg_idx = np.where(y_train == 0)[0][0]
print(decode_review(X_train[neg_idx])[:500], "...")

## Paso 5: Preprocesamiento — Padding de secuencias

Las redes neuronales requieren entradas de **longitud fija**. Dado que las reseñas tienen longitudes variables, aplicamos **padding** (relleno con ceros) o truncamiento para que todas tengan la misma longitud (`MAX_LEN`).

- **pre**: el padding/truncamiento se aplica al inicio de la secuencia
- **post**: al final (útil para CNNs)

Elegimos `MAX_LEN = 500`, que cubre el percentil 95 de las reseñas.

In [ ]:
# Hiperparámetro: longitud máxima de secuencia
MAX_LEN = 500

# Aplicamos padding/truncamiento
X_train_pad = pad_sequences(X_train, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test,  maxlen=MAX_LEN, padding='post', truncating='post')

print("=== Dimensiones después del padding ===")
print(f"X_train : {X_train_pad.shape}  → ({len(X_train)} muestras × {MAX_LEN} tokens)")
print(f"X_test  : {X_test_pad.shape}   → ({len(X_test)} muestras × {MAX_LEN} tokens)")
print(f"\nEjemplo de reseña después del padding (primeros 30 tokens):")
print(X_train_pad[0][:30])
print(f"\nEjemplo de reseña después del padding (últimos 30 tokens — zona de padding):")
print(X_train_pad[0][-30:])

## Paso 6: Arquitectura de la CNN-1D

### Componentes clave de la arquitectura:

| Capa | Descripción |
|------|-------------|
| **Embedding** | Convierte cada token (entero) en un vector denso de dimensión `EMBED_DIM`. Aprende representaciones distribuidas de palabras durante el entrenamiento. |
| **Conv1D** | Aplica filtros a ventanas de palabras consecutivas (n-gramas). Detecta patrones locales en el texto. |
| **BatchNormalization** | Normaliza las activaciones para estabilizar y acelerar el entrenamiento. |
| **GlobalMaxPooling1D** | Extrae el valor máximo de cada mapa de características, logrando invarianza a la posición. |
| **Dense + Dropout** | Capas completamente conectadas para la clasificación final. Dropout previene sobreajuste. |
| **Sigmoid** | Salida binaria: probabilidad de que la reseña sea positiva. |

In [ ]:
# ============================================================
#  HIPERPARÁMETROS DEL MODELO
# ============================================================
EMBED_DIM    = 128   # Dimensión del espacio de embeddings
NUM_FILTERS  = 128   # Número de filtros convolucionales
KERNEL_SIZE  = 5     # Tamaño del kernel (ventana de n-gramas)
DENSE_UNITS  = 64    # Unidades en la capa densa
DROPOUT_RATE = 0.5   # Tasa de dropout
LEARNING_RATE = 1e-3

# ============================================================
#  CONSTRUCCIÓN DEL MODELO
# ============================================================
def build_cnn1d_model(vocab_size, max_len, embed_dim, num_filters,
                      kernel_size, dense_units, dropout_rate):
    model = Sequential(name="CNN_1D_Sentiment")

    # --- Capa de Embedding ---
    # input_dim  = tamaño del vocabulario
    # output_dim = dimensión del vector de embedding
    # input_length = longitud de la secuencia de entrada
    model.add(Embedding(input_dim=vocab_size,
                        output_dim=embed_dim,
                        input_length=max_len,
                        name='embedding'))

    # --- Bloque Convolucional 1 ---
    model.add(Conv1D(filters=num_filters,
                     kernel_size=kernel_size,
                     activation='relu',
                     padding='same',
                     name='conv1d_1'))
    model.add(BatchNormalization(name='bn_1'))
    model.add(MaxPooling1D(pool_size=2, name='maxpool_1'))

    # --- Bloque Convolucional 2 ---
    model.add(Conv1D(filters=num_filters * 2,
                     kernel_size=kernel_size,
                     activation='relu',
                     padding='same',
                     name='conv1d_2'))
    model.add(BatchNormalization(name='bn_2'))
    model.add(MaxPooling1D(pool_size=2, name='maxpool_2'))

    # --- Bloque Convolucional 3 (captura patrones de alto nivel) ---
    model.add(Conv1D(filters=num_filters * 2,
                     kernel_size=kernel_size,
                     activation='relu',
                     padding='same',
                     name='conv1d_3'))
    model.add(BatchNormalization(name='bn_3'))

    # --- Pooling Global ---
    # Reduce la secuencia completa a un único vector de características
    model.add(GlobalMaxPooling1D(name='global_maxpool'))

    # --- Capas Densas de Clasificación ---
    model.add(Dense(dense_units, activation='relu', name='dense_1'))
    model.add(Dropout(dropout_rate, name='dropout_1'))
    model.add(Dense(32, activation='relu', name='dense_2'))
    model.add(Dropout(dropout_rate / 2, name='dropout_2'))

    # --- Capa de Salida ---
    # sigmoid → probabilidad de sentimiento positivo
    model.add(Dense(1, activation='sigmoid', name='output'))

    return model


# Instanciamos el modelo
model = build_cnn1d_model(
    vocab_size=VOCAB_SIZE,
    max_len=MAX_LEN,
    embed_dim=EMBED_DIM,
    num_filters=NUM_FILTERS,
    kernel_size=KERNEL_SIZE,
    dense_units=DENSE_UNITS,
    dropout_rate=DROPOUT_RATE
)

# Compilamos el modelo
optimizer = Adam(learning_rate=LEARNING_RATE)
model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Resumen de la arquitectura
model.summary()

## Paso 7: Entrenamiento del modelo

### Callbacks utilizados:
- **EarlyStopping**: Detiene el entrenamiento si `val_loss` no mejora después de `patience` épocas, y restaura los mejores pesos.
- **ReduceLROnPlateau**: Reduce la tasa de aprendizaje cuando el modelo deja de mejorar, permitiendo una convergencia más fina.

In [ ]:
# Hiperparámetros de entrenamiento
BATCH_SIZE = 128
EPOCHS     = 20

# Callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

# Entrenamiento
history = model.fit(
    X_train_pad, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.2,    # 20% del training como validación
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print(f"\nEntrenamiento finalizado en {len(history.history['loss'])} épocas.")

## Paso 8: Visualización del historial de entrenamiento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history.history['loss']) + 1)

# --- Gráfica de Pérdida ---
axes[0].plot(epochs_range, history.history['loss'],     'o-', color='steelblue',  label='Entrenamiento')
axes[0].plot(epochs_range, history.history['val_loss'], 's-', color='darkorange', label='Validación')
axes[0].set_title('Pérdida (Binary Cross-Entropy)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# --- Gráfica de Exactitud ---
axes[1].plot(epochs_range, history.history['accuracy'],     'o-', color='steelblue',  label='Entrenamiento')
axes[1].plot(epochs_range, history.history['val_accuracy'], 's-', color='darkorange', label='Validación')
axes[1].set_title('Exactitud (Accuracy)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Historial de Entrenamiento — CNN-1D Análisis de Sentimientos',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Paso 9: Evaluación del modelo en el conjunto de prueba

In [ ]:
# Evaluación general
test_loss, test_acc = model.evaluate(X_test_pad, y_test, verbose=0)
print(f"{'='*45}")
print(f"  Pérdida en test  : {test_loss:.4f}")
print(f"  Exactitud en test: {test_acc * 100:.2f}%")
print(f"{'='*45}")

# Predicciones
y_pred_prob = model.predict(X_test_pad, verbose=0).flatten()
y_pred      = (y_pred_prob >= 0.5).astype(int)

print("\n=== Reporte de Clasificación ===")
print(classification_report(y_test, y_pred,
                             target_names=['Negativo (0)', 'Positivo (1)']))

## Paso 10: Matriz de Confusión y Curva ROC

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Matriz de Confusión ---
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negativo', 'Positivo'],
            yticklabels=['Negativo', 'Positivo'],
            ax=axes[0], linewidths=0.5)
axes[0].set_title('Matriz de Confusión', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicción')
axes[0].set_ylabel('Valor Real')

# Anotaciones adicionales en la matriz
tn, fp, fn, tp = cm.ravel()
print(f"Verdaderos Negativos (TN): {tn}")
print(f"Falsos Positivos     (FP): {fp}")
print(f"Falsos Negativos     (FN): {fn}")
print(f"Verdaderos Positivos (TP): {tp}")

# --- Curva ROC ---
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
roc_auc = auc(fpr, tpr)

axes[1].plot(fpr, tpr, color='steelblue', lw=2,
             label=f'Curva ROC (AUC = {roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Clasificador aleatorio')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='steelblue')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_title('Curva ROC', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Tasa de Falsos Positivos (FPR)')
axes[1].set_ylabel('Tasa de Verdaderos Positivos (TPR)')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"\nÁrea bajo la curva ROC (AUC): {roc_auc:.4f}")

## Paso 11: Inferencia — Predicción sobre nuevas reseñas

Creamos una función de predicción que toma texto en inglés crudo, lo tokeniza y predice su sentimiento.

In [ ]:
def predict_sentiment(text, model, word_index, max_len=MAX_LEN, vocab_size=VOCAB_SIZE):
    """
    Predice el sentimiento de una reseña de texto en inglés.

    Parámetros:
        text      : str — Reseña en inglés
        model     : Modelo Keras entrenado
        word_index: Diccionario palabra→índice del dataset IMDB
        max_len   : Longitud máxima de secuencia (padding)
        vocab_size: Tamaño del vocabulario

    Retorna:
        Diccionario con la etiqueta, probabilidad y barra de confianza.
    """
    import re
    # Preprocesamiento básico
    text_clean = re.sub(r'[^a-z\s]', '', text.lower())
    tokens = text_clean.split()

    # Convertimos palabras a índices (desconocidas → 2 = <UNK>)
    # Los índices en IMDB están desplazados +3
    encoded = [word_index.get(w, 2) + 3 for w in tokens
               if word_index.get(w, 0) < vocab_size]

    # Padding
    padded = pad_sequences([encoded], maxlen=max_len,
                           padding='post', truncating='post')

    # Predicción
    prob = model.predict(padded, verbose=0)[0][0]
    label = 'POSITIVO 😊' if prob >= 0.5 else 'NEGATIVO 😞'

    return {'label': label, 'probabilidad': float(prob),
            'confianza': max(prob, 1 - prob)}


# ============================================================
# Ejemplos de prueba
# ============================================================
test_reviews = [
    "This movie was absolutely amazing! The performances were outstanding "
    "and the story kept me on the edge of my seat the entire time.",

    "Terrible film. The plot made no sense and the acting was laughably bad. "
    "Complete waste of time and money, I want my two hours back.",

    "A decent movie with some good moments but also quite a few boring parts. "
    "The ending was satisfying though.",

    "One of the worst films I have ever seen. Avoid at all costs. "
    "The director clearly had no idea what he was doing.",

    "A masterpiece of modern cinema. Beautiful cinematography, "
    "compelling characters and an unforgettable soundtrack."
]

print("=" * 65)
print("PREDICCIONES SOBRE NUEVAS RESEÑAS")
print("=" * 65)
for i, review in enumerate(test_reviews, 1):
    result = predict_sentiment(review, model, word_index)
    print(f"\n[Reseña {i}]")
    print(f"  Texto     : {review[:80]}...")
    print(f"  Sentimiento : {result['label']}")
    print(f"  Probabilidad positiva : {result['probabilidad']:.4f}")
    print(f"  Confianza             : {result['confianza'] * 100:.1f}%")

## Paso 12: Análisis de los filtros — ¿Qué aprende la CNN?

Visualizamos la distribución de los pesos aprendidos por la primera capa convolucional para entender qué tipo de patrones está capturando.

In [ ]:
# Extraemos los pesos de la primera capa convolucional
conv_weights = model.get_layer('conv1d_1').get_weights()
kernels = conv_weights[0]  # shape: (kernel_size, embed_dim, num_filters)
biases  = conv_weights[1]  # shape: (num_filters,)

print(f"Forma de los kernels : {kernels.shape}")
print(f"  → {kernels.shape[2]} filtros de tamaño {kernels.shape[0]} × {kernels.shape[1]}")
print(f"Forma de los sesgos  : {biases.shape}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Distribución de todos los pesos del kernel
axes[0].hist(kernels.flatten(), bins=60, color='steelblue',
             edgecolor='white', alpha=0.85)
axes[0].set_title('Distribución de pesos — Conv1D Capa 1', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Valor del peso')
axes[0].set_ylabel('Frecuencia')
axes[0].grid(True, alpha=0.3)

# Magnitud promedio por filtro (norma L2)
filter_norms = np.linalg.norm(kernels, axis=(0, 1))  # una norma por filtro
axes[1].bar(range(len(filter_norms)), sorted(filter_norms, reverse=True),
            color='darkorange', alpha=0.85)
axes[1].set_title('Norma L2 de cada filtro (ordenada)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Filtro (ordenado por magnitud)')
axes[1].set_ylabel('Norma L2')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nNorma media de los filtros : {filter_norms.mean():.4f}")
print(f"Filtro con mayor norma     : Filtro #{np.argmax(filter_norms)} ({filter_norms.max():.4f})")

## Paso 13: Comparación de hiperparámetros — Tamaño del kernel

El tamaño del kernel (ventana de n-gramas) es un hiperparámetro crítico. Experimentamos con diferentes valores para ver su efecto en el rendimiento.

In [ ]:
results = {}
kernel_sizes_to_test = [3, 5, 7]

for ks in kernel_sizes_to_test:
    print(f"\n{'='*50}")
    print(f"  Entrenando CNN-1D con kernel_size = {ks}")
    print(f"{'='*50}")

    m = build_cnn1d_model(
        vocab_size=VOCAB_SIZE, max_len=MAX_LEN, embed_dim=EMBED_DIM,
        num_filters=NUM_FILTERS, kernel_size=ks,
        dense_units=DENSE_UNITS, dropout_rate=DROPOUT_RATE
    )
    m.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
              loss='binary_crossentropy', metrics=['accuracy'])

    hist = m.fit(
        X_train_pad, y_train,
        batch_size=BATCH_SIZE,
        epochs=10,
        validation_split=0.2,
        callbacks=[EarlyStopping(monitor='val_loss', patience=3,
                                  restore_best_weights=True, verbose=0)],
        verbose=0
    )

    _, test_accuracy = m.evaluate(X_test_pad, y_test, verbose=0)
    best_val_acc = max(hist.history['val_accuracy'])
    results[ks] = {'test_acc': test_accuracy, 'val_acc': best_val_acc,
                   'epochs': len(hist.history['loss'])}
    print(f"  Test Accuracy: {test_accuracy:.4f} | Val Accuracy: {best_val_acc:.4f} "
          f"| Épocas: {len(hist.history['loss'])}")

# Tabla comparativa
print("\n=== TABLA COMPARATIVA ===")
df_results = pd.DataFrame(results).T
df_results.index.name = 'kernel_size'
df_results.columns = ['Test Accuracy', 'Val Accuracy', 'Épocas']
df_results['Test Accuracy'] = df_results['Test Accuracy'].map('{:.4f}'.format)
df_results['Val Accuracy']  = df_results['Val Accuracy'].map('{:.4f}'.format)
print(df_results.to_string())

## Conclusiones

En esta práctica hemos implementado exitosamente una **Red Neuronal Convolucional 1D (CNN-1D)** para el análisis de sentimientos sobre el dataset IMDB. A continuación se resumen los hallazgos principales:

### Resultados obtenidos
- El modelo CNN-1D alcanzó una exactitud competitiva en el conjunto de prueba.
- La capa de **Embedding** aprendió representaciones vectoriales de palabras adaptadas a la tarea de análisis de sentimientos.
- Los filtros convolucionales capturaron eficientemente **n-gramas locales** (secuencias de 3-7 palabras) que son discriminativos para determinar el sentimiento.
- El uso de **GlobalMaxPooling1D** permitió extraer las características más relevantes independientemente de su posición en la reseña.

### Ventajas de CNN-1D para NLP
- **Velocidad**: Las operaciones de convolución son paralelizables, lo que hace el entrenamiento significativamente más rápido que en RNNs/LSTMs.
- **Eficiencia**: Requiere menos parámetros que modelos de atención completa para textos cortos y medianos.
- **Interpretabilidad**: Los filtros pueden interpretarse como detectores de patrones lingüísticos (n-gramas con sentimiento).

### Reflexión sobre los hiperparámetros
- El **tamaño del kernel** (ventana de n-gramas) impacta directamente en la capacidad de capturar contexto: kernels pequeños (3) detectan bigramas/trigramas locales, mientras que kernels grandes (7) capturan frases más largas.
- El **dropout** fue esencial para prevenir el sobreajuste, dado el tamaño relativamente pequeño del vocabulario codificado.

### Trabajo futuro
- Comparar con arquitecturas LSTM/BiLSTM para secuencias largas.
- Usar embeddings preentrenados (Word2Vec, GloVe, FastText) en lugar de aprender los embeddings desde cero.
- Explorar arquitecturas híbridas CNN + RNN para combinar detección local con contexto global.

## Referencias

1. Yoon Kim (2014). *Convolutional Neural Networks for Sentence Classification*. EMNLP 2014.
2. Maas, A. L., et al. (2011). *Learning Word Vectors for Sentiment Analysis*. ACL 2011.
3. Chollet, F. (2021). *Deep Learning with Python* (2nd ed.). Manning Publications.
4. Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*. MIT Press.
5. TensorFlow/Keras Documentation: https://www.tensorflow.org/api_docs/python/tf/keras